# 13 - Final Test Evaluation

The one evaluation on the 5 held-out patients (records 116, 118, 119, 209, 215). This is the only notebook that loads them.

**What was fixed beforehand** (`results/metrics/frozen_protocol.json`, committed in `10c7825`, with the rules pre-registered in `6597cdf`): the logistic regression, its 13 features, the 0.5 threshold, its settings, and the split (whose SHA-256 is checked below). Nothing here changes any of it. **Whatever the result is, it is reported as it is** - the split is not redrawn, the features are not re-selected, and the threshold is not re-tuned on these patients.

**What is reported:** the frozen model's pooled and per-patient results with patient-level uncertainty (only 5 patients, so the intervals are wide), a first error analysis, and - for context only - the random forest and gradient boosting under their frozen settings. Those two cannot change the selection.

In [ ]:
import sys
sys.path.append("..")

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from src.evaluation import (annotated_rhythm, flagged_rate_by_symbol, metrics_by_group, patient_bootstrap_metrics,
                            summarize)
from src.feature_extraction import MODEL_FEATURES, add_record_relative_features, get_beat_waveform, load_beat_table
from src.models import make_gradient_boosting, make_logistic_regression, make_random_forest
from src.splitting import DEFAULT_SPLIT_PATH, apply_split, check_split, load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# 1. the frozen protocol must match what is about to be used
frozen = json.loads(Path("../results/metrics/frozen_protocol.json").read_text(encoding="utf-8"))
assert hashlib.sha256(Path(DEFAULT_SPLIT_PATH).read_bytes()).hexdigest() == frozen["split_file_sha256"], "split changed since the freeze"
assert frozen["selected_model"] == "logistic regression"
assert frozen["features"] == MODEL_FEATURES and frozen["feature_set"] == "13 baseline"
assert frozen["threshold"] == 0.5
THRESHOLD = frozen["threshold"]

# 2. data: train = the 12 development patients, test = the 5 held-out patients
table = add_record_relative_features(load_beat_table()).sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
data = apply_split(table, load_split())
check_split(data)
train, test = data[data.split == "train"].copy(), data[data.split == "test"].copy()
del table, data
print("protocol verified: split hash matches, model / features / threshold as frozen")
print(f"train: {len(train)} beats, {train.record.nunique()} patients ({sorted(train.record.unique())})")
print(f"test:  {len(test)} beats, {test.record.nunique()} patients ({sorted(test.record.unique())}); "
      f"{100 * test.is_abnormal.mean():.1f}% abnormal")
print("patients in both:", sorted(set(train.record) & set(test.record)))

## Training and prediction

The frozen logistic regression is fitted on **all 12 development patients** and predicts the test beats once. (The random forest and boosting are fitted the same way, only for the context comparison.)

In [ ]:
model = make_logistic_regression().fit(train[MODEL_FEATURES], train.is_abnormal)
assert model[-1].C == frozen["model_settings"]["C"] and model[-1].max_iter == frozen["model_settings"]["max_iter"]
proba = pd.Series(model.predict_proba(test[MODEL_FEATURES])[:, 1], index=test.index, name="proba")

point = summarize(test.is_abnormal, proba, THRESHOLD)
print(f"Frozen logistic regression on the 5 test patients (threshold {THRESHOLD}):")
print(f"  accuracy {point['accuracy']:.3f} | precision {point['precision']:.3f} | recall {point['recall']:.3f} | F1 {point['f1']:.3f}")
print(f"  false-alarm rate {point['false_alarm_rate']:.3f} | ROC-AUC {point['roc_auc']:.3f} | PR-AUC {point['pr_auc']:.3f}")
print(f"  confusion matrix: {point['tn']:,} true Normal, {point['fp']:,} false alarms, {point['fn']:,} missed abnormal, {point['tp']:,} caught abnormal")

### Pooled result with patient-level uncertainty

The 95% interval comes from resampling the 5 test patients with replacement (2,000 times). With only 5 patients there are few distinct resamples, so the interval is coarse and honest rather than tight. The last column is the development cross-validation score recorded at the freeze (out-of-fold on 12 patients), for comparison.

In [ ]:
boot = patient_bootstrap_metrics(test, proba, n_boot=2000, seed=0, threshold=THRESHOLD)
dev_cv = frozen["development_cross_validation_of_selected_model"]

rows = []
for metric in ["precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc", "accuracy"]:
    low, high = np.nanpercentile(boot[metric], [2.5, 97.5])
    rows.append({"metric": metric, "test (5 patients)": round(point[metric], 3),
                 "95% interval (patient bootstrap)": f"{low:.3f} to {high:.3f}",
                 "development CV (12 patients)": dev_cv[metric]})
result_table = pd.DataFrame(rows).set_index("metric")
print(f"{len(boot)} valid resamples. Reference: 'always predict Normal' scores accuracy {1 - test.is_abnormal.mean():.3f}; "
      f"a model with no skill has PR-AUC {test.is_abnormal.mean():.3f}.")
result_table

**At its frozen threshold, the frozen model is not good on these five patients - and the pooled numbers are not one story.**

- **At threshold 0.5:** recall 0.91 (1,093 of 1,204 abnormal beats caught) but precision only 0.17 and a false-alarm rate of 44% (5,174 of 11,752 Normal beats flagged). Accuracy, 0.59, is *below* the 0.907 that never flagging anything would score.
- **The threshold-free metrics look different:** ROC-AUC 0.89 (development cross-validation 0.76) and PR-AUC 0.68 (0.68 in development - and test prevalence is lower, 9.3% against 15.5%, so the same PR-AUC is relatively better). The model ranks beats reasonably; what fails is where the 0.5 line falls.
- **The 95% intervals are enormous** (precision 0.06 to 0.92, false-alarm rate 0.01 to 0.83). With five patients the result is dominated by *which* patients are in the sample, and the point estimates hide that two of them are near-perfect and two are flooded.

## Per patient

The pooled number averages over very different patients - here it matters even more with only five.

In [ ]:
columns = ["beats", "abnormal", "precision", "recall", "false_alarm_rate", "f1", "roc_auc", "pr_auc"]
per_patient = metrics_by_group(test, proba, "record")[columns].astype(float).round(3)
per_patient[["beats", "abnormal"]] = per_patient[["beats", "abnormal"]].astype(int)
per_patient

## Where the errors are

By beat type (recall for the abnormal types, false-alarm rate for the normal ones), and false alarms by annotated rhythm. Rhythm labels are for analysis only, never inputs.

In [ ]:
by_symbol = flagged_rate_by_symbol(test, proba, THRESHOLD)
print("Share of each beat type flagged as abnormal")
by_symbol

In [ ]:
rhythm = annotated_rhythm(test)
normal_test = test[test.label == "Normal"].assign(rhythm=rhythm, false_alarm=(proba >= THRESHOLD).astype(int))
by_rhythm = normal_test.groupby("rhythm").agg(normal_beats=("false_alarm", "size"), false_alarms=("false_alarm", "sum"))
by_rhythm["false_alarm_pct"] = (100 * by_rhythm.false_alarms / by_rhythm.normal_beats).round(1)
by_rhythm

## For context only: the other two models under their frozen settings

Fitted on the same 12 development patients, scored on the same 5 test patients. **They cannot change the selection** - the logistic regression was chosen by the pre-registered rule.

In [ ]:
context = {
    "logistic regression (frozen, selected)": proba,
    "random forest (context only)": pd.Series(make_random_forest().fit(train[MODEL_FEATURES], train.is_abnormal)
                                              .predict_proba(test[MODEL_FEATURES])[:, 1], index=test.index),
    "gradient boosting (context only)": pd.Series(make_gradient_boosting().fit(train[MODEL_FEATURES], train.is_abnormal)
                                                  .predict_proba(test[MODEL_FEATURES])[:, 1], index=test.index),
}
context_table = pd.DataFrame({name: summarize(test.is_abnormal, p, THRESHOLD) for name, p in context.items()}).T
context_table = context_table[["precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
context_table

In [ ]:
per_patient_context = pd.concat({name: metrics_by_group(test, p, "record")[["recall", "false_alarm_rate", "pr_auc"]].astype(float).round(3)
                                 for name, p in context.items()}, axis=1)
per_patient_context.columns = pd.MultiIndex.from_tuples([(n.split(" (")[0], c) for n, c in per_patient_context.columns])
print("Per test patient, all three models (threshold 0.5)")
per_patient_context

## Why were records 118 and 215 flooded with false alarms?

**Post-hoc analysis of the frozen model - it changes nothing.** Decompose the logistic regression's score (its log-odds) into per-feature contributions and compare each test patient's Normal beats with the training patients' Normal beats. A log-odds of 0 is the 0.5 threshold; the first table shows where each patient's typical Normal beat sits.

In [ ]:
clip_step, scaler_step, classifier = model[0], model[1], model[2]
z_train = pd.DataFrame(scaler_step.transform(clip_step.transform(train[MODEL_FEATURES])), index=train.index, columns=MODEL_FEATURES)
z_test = pd.DataFrame(scaler_step.transform(clip_step.transform(test[MODEL_FEATURES])), index=test.index, columns=MODEL_FEATURES)
weights = pd.Series(classifier.coef_[0], index=MODEL_FEATURES)
logit_train = z_train @ weights + classifier.intercept_[0]
logit_test = z_test @ weights + classifier.intercept_[0]

typical = {"training patients": logit_train[train.is_abnormal == 0].median()}
amplitude = {"training patients": train[train.is_abnormal == 0].amp_max.median()}
for record in sorted(test.record.unique()):
    normal_of_record = (test.record == record) & (test.is_abnormal == 0)
    typical[f"test {record}"] = logit_test[normal_of_record].median()
    amplitude[f"test {record}"] = test[normal_of_record].amp_max.median()
print("Typical Normal beat: log-odds of 'abnormal' (0 = the 0.5 threshold) and median R amplitude (amp_max, mV)")
pd.DataFrame({"median log-odds": pd.Series(typical).round(2), "median amp_max (mV)": pd.Series(amplitude).round(2)})

In [ ]:
train_normal_contribution = z_train[train.is_abnormal == 0].mul(weights).mean()
amplitude_features = ["amp_max", "amp_std", "qrs_p2p_mv"]
shift = {}
for record in ["118", "215", "116"]:
    part = test[(test.record == record) & (test.is_abnormal == 0)]
    contribution = z_test.loc[part.index].mul(weights).mean() - train_normal_contribution
    shift[f"test {record}"] = {**{f: contribution[f] for f in amplitude_features},
                               "all 10 other features": contribution.drop(amplitude_features).sum(), "total shift": contribution.sum()}
print("Shift in the average Normal beat's log-odds vs the training patients' Normal beats, by source")
print("(weights per +1 SD: " + ", ".join(f"{f} {weights[f]:+.1f}" for f in amplitude_features) + ")")
pd.DataFrame(shift).round(1)

In [ ]:
prevalence = test.is_abnormal.mean()
fig, axes = plt.subplots(1, 3, figsize=(20, 5.8), gridspec_kw={"width_ratios": [1, 1.1, 1]})

ax = axes[0]
styles = {"logistic regression (frozen, selected)": ("tab:blue", "-", 2.2), "random forest (context only)": ("tab:orange", "--", 1.2),
          "gradient boosting (context only)": ("tab:green", "--", 1.2)}
for name, p in context.items():
    color, style, width = styles[name]
    precision, recall, _ = precision_recall_curve(test.is_abnormal, p)
    s = summarize(test.is_abnormal, p, THRESHOLD)
    ax.plot(recall, precision, color=color, linestyle=style, linewidth=width, label=f"{name} (PR-AUC {s['pr_auc']:.2f})")
    if name.startswith("logistic"):
        ax.scatter(s["recall"], s["precision"], color=color, edgecolor="black", zorder=3, s=70, label="frozen threshold 0.5")
ax.axhline(prevalence, color="grey", linestyle=":", label=f"no skill ({prevalence:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Test patients: precision-recall")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
order = list(per_patient.index)
y = np.arange(len(order))
ax.barh(y - 0.2, per_patient.loc[order, "recall"], height=0.4, color="tab:red", label="recall (abnormal beats caught)")
ax.barh(y + 0.2, per_patient.loc[order, "false_alarm_rate"], height=0.4, color="tab:blue", label="false-alarm rate (normal beats)")
ax.set_yticks(y, [f"{r} ({100 * per_patient.loc[r, 'abnormal'] / per_patient.loc[r, 'beats']:.0f}% abn)" for r in order])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_title("Frozen model, per test patient (threshold 0.5)")
ax.legend(loc="lower right", fontsize=8)

ax = axes[2]
plot_symbols = by_symbol.sort_values("pct_flagged_abnormal")
ax.barh(plot_symbols.index, plot_symbols.pct_flagged_abnormal,
        color=["tab:red" if l == "Abnormal" else "tab:blue" for l in plot_symbols.label])
for i, (sym, row) in enumerate(plot_symbols.iterrows()):
    ax.text(row.pct_flagged_abnormal + 1, i, f"n={int(row.beats):,}", va="center", fontsize=8)
ax.set_xlim(0, 118)
ax.set_xlabel("% of beats flagged as abnormal")
ax.set_title("By beat type (red = truly abnormal: recall;\nblue = truly normal: false-alarm rate)", fontsize=10)

fig.tight_layout()
fig.savefig("../results/figures/25_test_evaluation.png", dpi=120)
plt.show()

### What do the mistakes look like?

A random sample (fixed seed 0, so not hand-picked) of abnormal beats the model missed (top row) and normal beats it flagged (bottom row), with the model's probability.

In [ ]:
rng = np.random.default_rng(0)
missed = test[(test.is_abnormal == 1) & (proba < THRESHOLD)]
alarms = test[(test.is_abnormal == 0) & (proba >= THRESHOLD)]
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(21, 7), sharex=True)
for row_axes, subset, title in [(axes[0], missed, "missed abnormal"), (axes[1], alarms, "false alarm")]:
    chosen = subset.loc[rng.choice(subset.index, size=min(n_show, len(subset)), replace=False)]
    for ax, (idx, beat) in zip(row_axes, chosen.iterrows()):
        t, w = get_beat_waveform(str(beat.record), int(beat.r_peak_sample))
        ax.plot(t, w, color="black", linewidth=1.1)
        ax.axvline(0, color="red", linestyle="--", linewidth=0.8)
        ax.set_title(f"{title}: rec {beat.record}, '{beat.symbol}'\nP(abnormal)={proba[idx]:.2f}  rr_pre={beat.rr_pre_s:.2f}s", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("ms from R peak")
axes[0][0].set_ylabel("mV")
axes[1][0].set_ylabel("mV")
fig.tight_layout()
fig.savefig("../results/figures/26_test_error_examples.png", dpi=110)
plt.show()
print(f"{len(missed):,} missed abnormal beats and {len(alarms):,} false alarms in total; {n_show} of each shown")

## Reading the errors

**Per patient - the pooled result is two very different stories:**

- Records **116 and 119 are near-perfect** (precision 0.96 and 1.00, recall 0.98 and 1.00, false alarms 0.2% and 0%).
- Records **118 and 215 are flooded with false alarms** (98.5% and 92.6% of their Normal beats flagged) - together **98% of all false alarms** (5,053 of 5,174).
- Record **209** is the honest middle: recall 0.74, false alarms 4.4%.
- **Ranking survives where the threshold does not:** 215's ROC-AUC is 0.98 and PR-AUC 0.96 despite a 93% false-alarm rate. The exception is 118 (ROC-AUC 0.50), where the model cannot rank its atrial beats above its normal ones.

**By beat type:** ventricular beats are caught 98.6% of the time, as in development. Atrial beats are caught **79%** - far above the ~10% seen in development cross-validation, a reminder that the development figure reflected which patients happened to hold the atrial beats (232, 220), not a fixed property of the model. And the right-bundle-branch beats (all from record 118) are flagged 98.5% of the time. **By rhythm:** every false alarm is in sinus rhythm; the fibrillation problem of the development set does not appear because none of these five patients is in fibrillation. The false alarms here are a *different* failure.

**Why 118 and 215 failed (the post-hoc decomposition above; it changes nothing).** The frozen model's predictions shift with a patient's overall signal amplitude. It leans on three strongly correlated amplitude features with large, opposite-sign weights (`amp_max` -9.8, `amp_std` +6.3, `qrs_p2p_mv` +4.2 per standard deviation - the weights notebook 09 flagged as unreadable). Record 215 is a low-amplitude patient (median R amplitude 0.95 mV against 1.90 in training): `amp_max` alone would lift its average Normal beat's log-odds by 14.7, partly cancelled by the other two amplitude features (-5.5 and -3.1), for a net shift of +6.0. Record 118 shifts by +9.3 (its typical Normal beat sits at +4.7, against -4.8 for the training patients). The same machinery pushes the *other* way for record 116, a high-amplitude patient (`amp_max` -17.2, partly cancelled by +5.6 and +7.7; typical log-odds -9.4), which is part of why it scores so well. The terms are large and only cancel if the three amplitude features move in the proportion seen in training - which a new patient's signal need not respect. Within a patient the ranking often survives, but the threshold does not transfer between patients. This is the failure mode the EDA (amplitude relationships change sign between patients), notebook 08 and notebook 09 warned about - here it is, measured, on unseen patients. Record 215 also has a fast resting rate (about 0.54 s between beats, against 0.74 in training), the absolute-interval confound noted earlier.

**The example beats** show what this looks like: the false alarms (bottom row) are ordinary-looking normal beats of record 215, and the misses (top row) are mostly record 209's atrial beats, which arrive early (0.41-0.52 s) but were given probabilities of only 0.12-0.47.

### The context models

For context only - they cannot change the selection - the random forest and boosting did **much** better on these patients: PR-AUC 0.93 and 0.94, F1 0.66 and 0.67 and a false-alarm rate of about 10%, against 0.68, 0.29 and 44% for the frozen logistic regression. They do not flood two patients the way the linear model does, but they have a failure of their own (per-patient table above): 32% (forest) and 24% (boosting) false alarms on record 116 - the highest-amplitude patient, 32% of whose beats lie outside the training range, where trees cannot extrapolate - exactly the record where the linear model scored a near-perfect 0.2%. Neither family is uniformly better; they fail on *different kinds of patient*. The overall gap is still large, and in the opposite direction to what development cross-validation suggested, where the three models were statistically indistinguishable.

The pre-registered rule chose the simplest model because the evidence could not separate them. It was a reasonable procedure that, on these five patients, chose the worse model. **Switching now would be wrong:** picking a model because it did better on the test set is exactly the leak the protocol exists to prevent, and the test set would stop being a test. The honest reading is that a 12-patient cross-validation was too coarse to see this, and that the tree models' advantage is a *hypothesis* to check on data not used so far.

## Saving the results

In [ ]:
def clean(value):
    return None if isinstance(value, float) and np.isnan(value) else (round(float(value), 4) if isinstance(value, (float, np.floating)) else value)


results = {
    "protocol": {"frozen_in_commit": "10c7825", "selected_model": frozen["selected_model"], "threshold": THRESHOLD,
                 "split_file_sha256": frozen["split_file_sha256"]},
    "test_patients": sorted(test.record.unique()),
    "test_beats": int(len(test)),
    "test_abnormal_share": round(float(test.is_abnormal.mean()), 4),
    "pooled": {k: clean(v) for k, v in point.items()},
    "pooled_95_interval_patient_bootstrap": {m: [clean(x) for x in np.nanpercentile(boot[m], [2.5, 97.5])] for m in boot.columns},
    "per_patient": {rec: {k: clean(v) for k, v in row.items()} for rec, row in per_patient.iterrows()},
    "context_only": {name: {k: clean(v) for k, v in row.items()} for name, row in context_table.iterrows()},
}
Path("../results/metrics/test_results.json").write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
print("saved results/metrics/test_results.json")

## Summary

**Frozen logistic regression on the 5 held-out patients (threshold 0.5):** recall 0.91, precision 0.17, F1 0.29, false-alarm rate 44%; ROC-AUC 0.89, PR-AUC 0.68 (prevalence 9.3%). With 5 patients the 95% intervals span most of the range: the result is a description, not a precise estimate.

**Verdict.** As an operating classifier at the frozen threshold, the model does **not** generalise across patients: 98% of its false alarms come from two patients whose signal amplitude differs from the training patients, because it depends on collinear absolute-amplitude features. As a *ranking* it is reasonable (ROC-AUC 0.89; near-perfect on records 116 and 119). Nothing about that is changed after the fact, and none of it is a clinical claim.

**What was learned about the method**

1. The pre-registration and the freeze did their job: a disappointing result is reported as it is, with no way to quietly adjust.
2. Development cross-validation over 12 patients could not separate the models, and the "prefer the simplest within noise" rule then selected one that turned out worse on unseen patients. Small numbers of patients are a real limit on model selection.
3. The warning signs were all present earlier (amplitude relationships changing sign between patients, unreadable collinear weights, per-patient variation dwarfing between-model variation) - they turned out to matter.

**What could legitimately follow: a version 2, judged on data not used so far, under new pre-registered rules.** Twelve recordings passed the detector quality gate but were never used because they have fewer than 50 abnormal beats (101, 103, 112, 113, 115, 117, 121, 122, 123, 212, 230, 231; record 100 is excluded because the detector was developed on it): about 24,000 Normal beats and only 23 abnormal ones. They cannot say much about recall, but they are a direct test of the failure found here - false-alarm flooding in new patients. Whatever is tried, this test set stays as the record of version 1.